# 01_07_cft_to_agr_active_checks

Диагностическая тетрадка для проверки гипотезы:

- может ли один `cft_id` иметь несколько **активных** `agr_id` в выбранном месяце;
- сколько таких кейсов, какая доля, и какие конкретно `cft_id` попадают в риск;
- есть ли обратные аномалии (`agr_id` -> несколько `cft_id`).

Тетрадка read-only: ничего не создает и не изменяет в Озере.

In [ ]:
from getpass import getpass

import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
# Конфиг проверки
report_month = '2026-05-01'  # первый день месяца
run_invalidate_metadata = True
show_top_n = 200

report_month_ts = pd.to_datetime(report_month)
month_start = report_month_ts.strftime('%Y-%m-%d')
month_end = (report_month_ts + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
report_month_label = report_month_ts.strftime('%Y-%m')

invalidate_tables = [
    'ods_alpha.scd1_agreements',
    'ods_alpha.scd1_companies',
    'ods_alpha.scd1_agr_terms',
    'ocrm_ul.s_org_ext',
    'cdiul.ext_id_org',
]

print('report_month =', report_month_label)
print('month_start =', month_start)
print('month_end =', month_end)

In [ ]:
# Подключение к Impala
if 'imp' in globals() and imp is not None:
    print('Using existing imp connection from current session')
else:
    imp = connect(
        to='IMPALA',
        extra_options={'db': 'sandbox_ai'},
        driver_args={'tez.queue.name': 'ai'},
        kerberos={
            'keytab_path': '/home/jovyan/test_requests/tech.keytab',
            'use_credentials': True,
            'update_keytab': True,
        },
        user_params={'user_name': 'Shestopalov-VYur'}
    )

try:
    imp._init_connection()
except Exception:
    pass

print('Impala connection initialized')

if run_invalidate_metadata:
    invalidate_ok = 0
    invalidate_failed = []
    with imp:
        for t in invalidate_tables:
            try:
                imp.execute(f'invalidate metadata {t}')
                invalidate_ok += 1
                print(f'[invalidate ok] {t}')
            except Exception as e:
                invalidate_failed.append((t, type(e).__name__, str(e)))
                print(f'[invalidate fail] {t}: {type(e).__name__}')

    print(f'Invalidate completed: ok={invalidate_ok}, failed={len(invalidate_failed)}')
    if invalidate_failed:
        display(pd.DataFrame(invalidate_failed, columns=['table_name', 'error_type', 'error_message']))
else:
    print('Invalidate skipped')

In [ ]:
# 1) Базовый датасет: активные SA-договоры в месяце + соответствующий cft_id
sql_cft_agr_active = f"""
with sa_agr as (
  select distinct
    cast(a.abs_agr_id as string) as agr_id,
    cast(a.n_agr as string) as n_agr,
    cast(a.n_cmp_client as string) as n_cmp_client,
    cast(a.c_agr_number as string) as contract_number,
    cast(a.d_valid_from as date) as d_valid_from,
    cast(a.d_valid_to as date) as d_valid_to,
    regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn
  from ods_alpha.scd1_agreements a
  join ods_alpha.scd1_companies c
    on c.n_cmp = a.n_cmp_client
  where upper(trim(cast(a.acq_class as string))) = 'SA'
    and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
    and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
    and coalesce(a.ods_deleted_flg, '0') <> '1'
    and coalesce(c.ods_deleted_flg, '0') <> '1'
    and c.c_inn is not null
    and exists (
      select 1
      from ods_alpha.scd1_agr_terms t
      where cast(t.n_agr as string) = cast(a.n_agr as string)
        and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
        and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
        and upper(trim(cast(t.cf_ter_type as string))) = 'P'
        and coalesce(t.ods_deleted_flg, '0') <> '1'
    )
),
inn_scope as (
  select distinct inn
  from sa_agr
  where inn is not null and trim(inn) <> ''
),
ocrm_current as (
  select
    regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') as inn,
    cast(soe.row_id as string) as row_id,
    row_number() over (
      partition by regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '')
      order by cast(soe.created as timestamp) desc, cast(soe.row_id as string) desc
    ) as rn
  from ocrm_ul.s_org_ext soe
  join inn_scope s
    on s.inn = regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '')
  where coalesce(soe.x_removed_flg, 'N') = 'N'
    and coalesce(soe.x_duplicate_flg, 'N') = 'N'
),
ocrm_one as (
  select inn, row_id
  from ocrm_current
  where rn = 1
),
cdi_map as (
  select
    o.inn,
    cast(e.party_id as string) as cdi_id
  from ocrm_one o
  left join cdiul.ext_id_org e
    on cast(e.cmo_ext_party_source_id as string) = o.row_id
   and upper(cast(e.cmo_ext_source_system as string)) like 'OCRM%'
),
cft_map as (
  select
    cast(e.party_id as string) as cdi_id,
    cast(e.cmo_ext_party_source_id as string) as cft_id
  from cdiul.ext_id_org e
  where upper(cast(e.cmo_ext_source_system as string)) like 'CFT%'
),
sa_with_cft as (
  select
    sa.agr_id,
    sa.n_agr,
    sa.n_cmp_client,
    sa.contract_number,
    sa.d_valid_from,
    sa.d_valid_to,
    sa.inn,
    cdi.cdi_id,
    cft.cft_id
  from sa_agr sa
  left join cdi_map cdi
    on cdi.inn = sa.inn
  left join cft_map cft
    on cft.cdi_id = cdi.cdi_id
)
select
  agr_id,
  n_agr,
  n_cmp_client,
  contract_number,
  d_valid_from,
  d_valid_to,
  inn,
  cdi_id,
  cft_id
from sa_with_cft
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    cft_agr_raw_df = imp.fetch(sql_cft_agr_active)

if cft_agr_raw_df is None:
    cft_agr_raw_df = pd.DataFrame(columns=['agr_id', 'n_agr', 'n_cmp_client', 'contract_number', 'd_valid_from', 'd_valid_to', 'inn', 'cdi_id', 'cft_id'])

for c in ['agr_id', 'n_agr', 'n_cmp_client', 'contract_number', 'inn', 'cdi_id', 'cft_id']:
    if c in cft_agr_raw_df.columns:
        cft_agr_raw_df[c] = cft_agr_raw_df[c].astype(str).str.strip()

cft_agr_raw_df = cft_agr_raw_df.replace({'': pd.NA, 'None': pd.NA, 'nan': pd.NA})

print('Rows in raw result:', len(cft_agr_raw_df))
print('Distinct agr_id:', cft_agr_raw_df['agr_id'].dropna().nunique())
print('Distinct cft_id:', cft_agr_raw_df['cft_id'].dropna().nunique())
display(cft_agr_raw_df.head(10))

In [ ]:
# 2) Основная проверка: сколько активных agr_id приходится на каждый cft_id
if cft_agr_raw_df is None or cft_agr_raw_df.empty:
    raise RuntimeError('Пустой результат. Проверьте входные таблицы/параметры месяца.')

cft_agr_df = (
    cft_agr_raw_df
    .dropna(subset=['cft_id', 'agr_id'])
    .drop_duplicates(subset=['cft_id', 'agr_id'])
    .copy()
)

if cft_agr_df.empty:
    raise RuntimeError('Нет строк с заполненными cft_id и agr_id для анализа.')

cft_stats_df = (
    cft_agr_df
    .groupby('cft_id', as_index=False)
    .agg(
        active_agr_cnt=('agr_id', 'nunique'),
        active_n_agr_cnt=('n_agr', 'nunique'),
        inn_cnt=('inn', 'nunique'),
        n_cmp_client_cnt=('n_cmp_client', 'nunique'),
    )
    .sort_values(['active_agr_cnt', 'inn_cnt', 'cft_id'], ascending=[False, False, True])
)

cft_distribution_df = (
    cft_stats_df
    .groupby('active_agr_cnt', as_index=False)
    .agg(cft_id_cnt=('cft_id', 'nunique'))
    .sort_values('active_agr_cnt')
)

cft_multi_df = cft_stats_df[cft_stats_df['active_agr_cnt'] > 1].copy()

summary_df = pd.DataFrame([
    {'metric': 'report_month', 'value': report_month_label},
    {'metric': 'distinct_cft_id', 'value': int(cft_stats_df['cft_id'].nunique())},
    {'metric': 'distinct_agr_id_linked_to_cft', 'value': int(cft_agr_df['agr_id'].nunique())},
    {'metric': 'cft_id_with_1_active_agr_id', 'value': int((cft_stats_df['active_agr_cnt'] == 1).sum())},
    {'metric': 'cft_id_with_2plus_active_agr_id', 'value': int((cft_stats_df['active_agr_cnt'] > 1).sum())},
    {'metric': 'share_cft_with_2plus_active_agr_id_pct', 'value': round(100.0 * (cft_stats_df['active_agr_cnt'] > 1).mean(), 2)},
    {'metric': 'max_active_agr_id_per_cft_id', 'value': int(cft_stats_df['active_agr_cnt'].max())},
])

print('Summary:')
display(summary_df)

print('Distribution (active agr_id per cft_id):')
display(cft_distribution_df)

print(f'Top {show_top_n} cft_id by active_agr_cnt:')
display(cft_stats_df.head(show_top_n))

In [ ]:
# 3) Детализация cft_id, где активных agr_id больше 1
if 'cft_multi_df' not in globals():
    raise RuntimeError('Сначала выполните секцию 2 с агрегированной статистикой.')

if cft_multi_df.empty:
    print('В выбранном месяце не найдено cft_id с несколькими активными agr_id.')
else:
    cft_multi_details_df = (
        cft_agr_df
        .groupby('cft_id', as_index=False)
        .agg(
            active_agr_cnt=('agr_id', 'nunique'),
            agr_id_list=('agr_id', lambda s: ', '.join(sorted(set(s.dropna().astype(str))))),
            n_agr_list=('n_agr', lambda s: ', '.join(sorted(set(s.dropna().astype(str))))),
            inn_list=('inn', lambda s: ', '.join(sorted(set(s.dropna().astype(str))))),
            n_cmp_client_list=('n_cmp_client', lambda s: ', '.join(sorted(set(s.dropna().astype(str))))),
        )
        .query('active_agr_cnt > 1')
        .sort_values(['active_agr_cnt', 'cft_id'], ascending=[False, True])
        .reset_index(drop=True)
    )

    print(f'Найдено cft_id с 2+ активными agr_id: {len(cft_multi_details_df):,}')
    display(cft_multi_details_df.head(show_top_n))

In [ ]:
# 4) Обратная проверка: есть ли agr_id, связанные сразу с несколькими cft_id
agr_cft_df = (
    cft_agr_df
    .dropna(subset=['agr_id', 'cft_id'])
    .drop_duplicates(subset=['agr_id', 'cft_id'])
    .copy()
)

agr_stats_df = (
    agr_cft_df
    .groupby('agr_id', as_index=False)
    .agg(
        cft_cnt=('cft_id', 'nunique'),
        cft_id_list=('cft_id', lambda s: ', '.join(sorted(set(s.dropna().astype(str))))),
        n_cmp_client_cnt=('n_cmp_client', 'nunique'),
        inn_cnt=('inn', 'nunique'),
    )
    .sort_values(['cft_cnt', 'agr_id'], ascending=[False, True])
)

agr_multi_cft_df = agr_stats_df[agr_stats_df['cft_cnt'] > 1].copy()

reverse_summary_df = pd.DataFrame([
    {'metric': 'distinct_agr_id_in_mapping', 'value': int(agr_stats_df['agr_id'].nunique())},
    {'metric': 'agr_id_with_2plus_cft_id', 'value': int((agr_stats_df['cft_cnt'] > 1).sum())},
    {'metric': 'share_agr_with_2plus_cft_pct', 'value': round(100.0 * (agr_stats_df['cft_cnt'] > 1).mean(), 2)},
    {'metric': 'max_cft_per_agr_id', 'value': int(agr_stats_df['cft_cnt'].max())},
])

print('Reverse mapping summary (agr_id -> cft_id):')
display(reverse_summary_df)

if not agr_multi_cft_df.empty:
    print(f'Top {show_top_n} agr_id with multiple cft_id:')
    display(agr_multi_cft_df.head(show_top_n))
else:
    print('agr_id with multiple cft_id not found in current month.')

In [ ]:
# 5) Детализация по выбранным cft_id
# Можно заполнить вручную: selected_cft_ids = ['123', '456']
selected_cft_ids = []

if not selected_cft_ids and 'cft_multi_df' in globals() and not cft_multi_df.empty:
    selected_cft_ids = cft_multi_df['cft_id'].head(10).astype(str).tolist()

if not selected_cft_ids:
    print('Нет selected_cft_ids для детализации.')
else:
    selected_cft_ids = [str(x).strip() for x in selected_cft_ids if str(x).strip()]
    details_df = (
        cft_agr_raw_df[cft_agr_raw_df['cft_id'].astype(str).isin(selected_cft_ids)]
        .sort_values(['cft_id', 'agr_id', 'd_valid_from'], ascending=[True, True, False])
        .reset_index(drop=True)
    )

    print('selected_cft_ids =', selected_cft_ids)
    print('Rows in details_df =', len(details_df))
    display(details_df.head(500))